# Tarea 13 - Agente RAG

### Tema principal: Energías renovables
### Tema oculto: Aeronaves comerciales

___

___

## Dependencias

In [27]:
from pathlib import Path

import ollama
import chromadb
import pymupdf
import json
from ddgs import DDGS

from sentence_transformers import SentenceTransformer

___

___

## Configuración

In [28]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Paths
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ROOT = Path.cwd()
DOCS_DIR = ROOT / "docs"
CHROMA_DIR = ROOT / "vectors"
COLLECTION_NAME = "nlp_docs"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Modelos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OLLAMA_MODEL = "granite4.1:3b"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Chunking
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CHUNK_SIZE = 700
CHUNK_OVERLAP = 150

___

___

## Sistema RAG

### Documentos

In [29]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Carga de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_pdf_documents(docs_dir: Path) -> list[dict]:
    documents = []

    pdf_files = sorted(docs_dir.glob("*.pdf"))

    for pdf_path in pdf_files:
        pdf_document = pymupdf.open(pdf_path)

        for page_index in range(len(pdf_document)):
            page = pdf_document[page_index]

            text = page.get_text().strip()

            if text:
                documents.append(
                    {
                        "source": pdf_path.name,
                        "page": page_index + 1,
                        "text": text
                    }
                )

        pdf_document.close()

    return documents

documents = load_pdf_documents(DOCS_DIR)

In [30]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Chunking de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

def create_pdf_chunks(documents: list[dict]) -> list[dict]:
    chunks = []

    for document in documents:
        text_chunks = split_text(
            text=document["text"],
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        for chunk_index, chunk_text in enumerate(text_chunks):
            chunk_id = f"{document['source']}_page_{document['page']}_chunk_{chunk_index}"

            chunks.append(
                {
                    "id": chunk_id,
                    "source": document["source"],
                    "page": document["page"],
                    "chunk_index": chunk_index,
                    "text": chunk_text
                }
            )

    return chunks

chunks = create_pdf_chunks(documents)

print("Chunks creados:", len(chunks))

for chunk in chunks[:3]:
    print("ID:", chunk["id"])
    print("SOURCE:", chunk["source"])
    print("PAGE:", chunk["page"])
    print("TEXT:", chunk["text"][:500])
    print("-" * 80)

Chunks creados: 2578
ID: 27955.pdf_page_1_chunk_0
SOURCE: 27955.pdf
PAGE: 1
TEXT: What is Renewable Energy?
Renewable energy uses energy sources
that are continually replenished by
nature—the sun, the wind, water, the
Earth’s heat, and plants. Renewable
energy technologies turn these fuels into
usable forms of energy—most often elec-
tricity, but also heat, chemicals, or
mechanical power.
Why Use Renewable Energy?
Today we primarily use fossil fuels to heat
and power our homes and fuel our cars.
It’s convenient to use coal, oil, and natural
gas for meeting our energy needs, b
--------------------------------------------------------------------------------
ID: 27955.pdf_page_1_chunk_1
SOURCE: 27955.pdf
PAGE: 1
TEXT: Earth. We’re using them much more
rapidly than they are being created. Even-
tually, they will run out. And because of 
safety concerns and waste disposal prob-
lems, the United States will retire much of
its nuclear capacity by 2020. In the mean-
time, the nation’s energy n

### ChromaDB

In [31]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Cliente de ChromaDB
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Colección de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME
)



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Indexación de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def index_pdf_chunks(collection, chunks: list[dict], embedding_model) -> None:
    ids = []
    documents = []
    metadatas = []
    embeddings = []

    for chunk in chunks:
        ids.append(chunk["id"])
        documents.append(chunk["text"])

        metadatas.append(
            {
                "source": chunk["source"],
                "page": chunk["page"],
                "chunk_index": chunk["chunk_index"]
            }
        )

        embedding = embedding_model.encode(chunk["text"]).tolist()

        embeddings.append(embedding)

    if ids:
        collection.upsert(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings
        )


embedding_model = SentenceTransformer(EMBEDDING_MODEL)
index_pdf_chunks(
    collection=collection,
    chunks=chunks,
    embedding_model=embedding_model
)

print("Chunks indexados en Chroma:", collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21379.39it/s]


Chunks indexados en Chroma: 2578


### RAG Tool

In [32]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Búsqueda de documentos relevantes
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def search_pdf_rag(query: str, top_k: int = 3) -> list[dict]:
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_chunks = []

    if not results["ids"] or not results["ids"][0]:
        return retrieved_chunks

    for index in range(len(results["ids"][0])):
        retrieved_chunks.append(
            {
                "id": results["ids"][0][index],
                "text": results["documents"][0][index],
                "metadata": results["metadatas"][0][index],
                "distance": results["distances"][0][index]
            }
        )

    return retrieved_chunks


def build_pdf_context(retrieved_chunks: list[dict]) -> str:
    context_parts = []

    for chunk in retrieved_chunks:
        source = chunk["metadata"]["source"]
        page = chunk["metadata"]["page"]
        text = chunk["text"]

        context_parts.append(
            f"Fuente: {source}, página {page}\nContenido:\n{text}"
        )

    return "\n\n".join(context_parts)

In [33]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Definición de la tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RAG_TOOL_DEFINITION = {
    "name": "rag_search_tool",
    "description": (
        "Busca información relevante en los documentos PDF locales "
        "usando Chroma y embeddings con all-MiniLM-L6-v2."
    ),
    "input": {
        "query": "Pregunta o consulta del usuario en lenguaje natural.",
        "top_k": "Número de fragmentos relevantes a recuperar."
    },
    "output": {
        "tool_name": "Nombre de la herramienta ejecutada.",
        "found": "Indica si se encontró información potencialmente útil.",
        "context": "Contexto construido con los chunks recuperados.",
        "sources": "Fuentes PDF, páginas, chunk index y distancia.",
        "best_distance": "Distancia del resultado más relevante."
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def rag_search_tool(query: str, top_k: int = 3) -> dict:
    retrieved_chunks = search_pdf_rag(
        query=query,
        top_k=top_k
    )

    if not retrieved_chunks:
        return {
            "tool_name": "rag_search_tool",
            "found": False,
            "context": "",
            "sources": [],
            "used_chunks": [],
            "best_distance": None,
            "reason": "No se recuperaron chunks desde Chroma."
        }

    context = build_pdf_context(retrieved_chunks)

    sources = []
    used_chunks = []

    for chunk in retrieved_chunks:
        source = chunk["metadata"]["source"]
        page = chunk["metadata"]["page"]
        chunk_index = chunk["metadata"]["chunk_index"]
        distance = chunk["distance"]
        text = chunk["text"]

        sources.append(
            {
                "source": source,
                "page": page,
                "chunk_index": chunk_index,
                "distance": distance
            }
        )

        used_chunks.append(
            {
                "source": source,
                "page": page,
                "chunk_index": chunk_index,
                "distance": distance,
                "text": text
            }
        )

    best_distance = min(
        chunk["distance"] for chunk in retrieved_chunks
    )

    return {
        "tool_name": "rag_search_tool",
        "found": True,
        "context": context,
        "sources": sources,
        "used_chunks": used_chunks,
        "best_distance": best_distance,
        "reason": "Se recuperaron chunks desde los PDFs locales."
    }

___

___

## Web Search

In [34]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Definición de la tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WEB_TOOL_DEFINITION = {
    "name": "web_search_tool",
    "description": (
        "Busca información en internet usando DuckDuckGo mediante ddgs. "
        "Debe utilizarse cuando el sistema RAG local no contiene información suficiente "
        "para responder la pregunta del usuario."
    ),
    "input": {
        "query": "Pregunta o consulta del usuario en lenguaje natural.",
        "max_results": "Número máximo de resultados de búsqueda a recuperar."
    },
    "output": {
        "tool_name": "Nombre de la herramienta ejecutada.",
        "found": "Indica si se encontraron resultados en internet.",
        "context": "Texto construido a partir de los resultados de búsqueda.",
        "sources": "Lista de URLs recuperadas.",
        "reason": "Explicación breve del resultado de la búsqueda."
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Web Search tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def web_search_tool(query: str, max_results: int = 5) -> dict:

    try:
        results = list(
            DDGS().text(
                query=query,
                max_results=max_results
            )
        )

        if not results:
            return {
                "tool_name": "web_search_tool",
                "found": False,
                "context": "",
                "sources": [],
                "reason": "No se encontraron resultados en internet."
            }

        context_parts = []
        sources = []

        for result in results:
            title = result.get("title", "")
            body = result.get("body", "")
            href = result.get("href", "")

            context_parts.append(
                f"Título: {title}\nContenido: {body}\nURL: {href}"
            )

            if href:
                sources.append(href)

        context = "\n\n".join(context_parts)

        return {
            "tool_name": "web_search_tool",
            "found": True,
            "context": context,
            "sources": sources,
            "reason": "Se encontró información en internet usando ddgs."
        }

    except Exception as error:
        return {
            "tool_name": "web_search_tool",
            "found": False,
            "context": "",
            "sources": [],
            "reason": f"Ocurrió un error al buscar en internet: {error}"
        }

___

___

## Evaluador de contexto

In [35]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Definición de la tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RAG_EVALUATOR_TOOL_DEFINITION = {
    "name": "evaluate_rag_context",
    "description": (
        "Evalúa si el contexto recuperado por el sistema RAG contiene "
        "información suficiente para responder la pregunta del usuario."
    ),
    "input": {
        "query": "Pregunta original del usuario.",
        "context": "Contexto recuperado desde los documentos PDF locales."
    },
    "output": {
        "tool_name": "Nombre de la herramienta ejecutada.",
        "is_enough": "True si el contexto es suficiente, False si no lo es.",
        "reason": "Explicación breve de la decisión."
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Tool para evaluar el contexto del RAG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def evaluate_rag_context(query: str, context: str) -> dict:
    if not context.strip():
        return {
            "tool_name": "evaluate_rag_context",
            "is_enough": False,
            "reason": "El contexto recuperado desde el RAG está vacío."
        }

    system_prompt = """
    Eres un evaluador de contexto para un sistema RAG.

    Tu tarea es decidir si el contexto proporcionado contiene información suficiente
    para responder la pregunta del usuario.

    Reglas:
    - No respondas la pregunta del usuario.
    - Solo evalúa si el contexto es suficiente.
    - Si el contexto contiene una respuesta clara a la pregunta, responde true.
    - Si el contexto habla de otro tema, responde false.
    - Si el contexto solo menciona palabras relacionadas, pero no responde realmente, responde false.
    - Responde únicamente en formato JSON válido.

    El JSON debe tener exactamente esta estructura:
    {
        "is_enough": true,
        "reason": "explicación breve"
    }
    """

    user_prompt = f"""
    Pregunta del usuario:
    {query}

    Contexto recuperado desde el RAG:
    {context}
    """

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    raw_response = response["message"]["content"].strip()

    try:
        evaluation = json.loads(raw_response)

        return {
            "tool_name": "evaluate_rag_context",
            "is_enough": bool(evaluation.get("is_enough", False)),
            "reason": evaluation.get("reason", "No se proporcionó una razón.")
        }

    except json.JSONDecodeError:
        return {
            "tool_name": "evaluate_rag_context",
            "is_enough": False,
            "reason": f"El evaluador no devolvió JSON válido: {raw_response}"
        }

___

___

## Agente

### System prompt

In [36]:
AGENT_SYSTEM_PROMPT = """
Eres un agente de preguntas y respuestas especializado en Procesamiento de Lenguaje Natural.

Tu objetivo es responder preguntas del usuario usando la mejor fuente disponible.

Reglas principales:
1. Siempre debes intentar usar primero la información recuperada desde el sistema RAG local.
2. Si el sistema RAG contiene información suficiente, responde usando únicamente ese contexto.
3. Si el sistema RAG no contiene información suficiente, utiliza información recuperada desde internet.
4. No inventes información.
5. Si ninguna fuente contiene información suficiente, dilo claramente.
6. Cuando uses información del RAG, menciona las fuentes PDF y páginas si están disponibles.
7. Cuando uses información de internet, menciona que la información proviene de búsqueda web.
8. Responde de forma clara, ordenada y útil para un estudiante de Procesamiento de Lenguaje Natural.

No debes explicar el funcionamiento interno del agente a menos que el usuario lo pregunte.
"""

### Tool map

In [37]:
TOOL_MAP = {
    "rag_search_tool": {
        "description": "Busca información en documentos PDF locales usando Chroma.",
        "function": rag_search_tool
    },
    "evaluate_rag_context": {
        "description": "Evalúa si el contexto recuperado por RAG es suficiente.",
        "function": evaluate_rag_context
    },
    "web_search_tool": {
        "description": "Busca información en internet usando DuckDuckGo mediante ddgs.",
        "function": web_search_tool
    }
}

### Generador de respuesta

In [38]:
def generate_answer_with_context(query: str, context: str, source_type: str) -> str:
    if source_type == "rag":
        source_instruction = """
        La información proviene del sistema RAG local.
        Usa únicamente el contexto proporcionado.
        Menciona las fuentes PDF y páginas cuando estén disponibles.
        """

    elif source_type == "web":
        source_instruction = """
        La información proviene de una búsqueda web.
        Usa únicamente el contexto proporcionado.
        Indica que la información fue obtenida mediante búsqueda web.
        """

    else:
        source_instruction = """
        Usa únicamente el contexto proporcionado.
        """

    user_prompt = f"""
    Pregunta del usuario:
    {query}

    Tipo de fuente:
    {source_type}

    Instrucciones sobre la fuente:
    {source_instruction}

    Contexto disponible:
    {context}

    Respuesta:
    """

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": AGENT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response["message"]["content"]

### Formato para fuentes

In [39]:
def format_rag_sources(used_chunks: list[dict]) -> str:
    if not used_chunks:
        return "No se encontraron fuentes RAG."

    formatted_sources = []

    for index, chunk in enumerate(used_chunks, start=1):
        formatted_sources.append(
            f"""
Fuente {index}
PDF: {chunk["source"]}
Página: {chunk["page"]}
Chunk: {chunk["chunk_index"]}
Distancia: {chunk["distance"]}

Texto utilizado:
{chunk["text"]}
"""
        )

    return "\n".join(formatted_sources)

### Loop

In [40]:
def agent_loop(user_query: str, verbose: bool = True) -> dict:
    if verbose:
        print("Agent: buscando información en el RAG local...")

    rag_result = TOOL_MAP["rag_search_tool"]["function"](
        query=user_query,
        top_k=3
    )

    if verbose:
        print("Agent: evaluando si el contexto del RAG es suficiente...")

    rag_evaluation = TOOL_MAP["evaluate_rag_context"]["function"](
        query=user_query,
        context=rag_result["context"]
    )

    if rag_evaluation["is_enough"]:
        if verbose:
            print("Agent: el RAG tiene información suficiente.")
            print("Agent: generando respuesta final con contexto RAG...")

        answer = generate_answer_with_context(
            query=user_query,
            context=rag_result["context"],
            source_type="rag"
        )

        rag_sources_text = format_rag_sources(
            rag_result["used_chunks"]
        )

        return {
            "answer": answer,
            "source_type": "rag",
            "rag_result": rag_result,
            "rag_evaluation": rag_evaluation,
            "rag_sources_text": rag_sources_text,
            "web_result": None
        }

    if verbose:
        print("Agent: el RAG no tiene información suficiente.")
        print("Agent: buscando información en internet...")

    web_result = TOOL_MAP["web_search_tool"]["function"](
        query=user_query,
        max_results=5
    )

    if not web_result["found"]:
        return {
            "answer": (
                "No encontré información suficiente en los documentos locales "
                "ni en la búsqueda web para responder con confianza."
            ),
            "source_type": "none",
            "rag_result": rag_result,
            "rag_evaluation": rag_evaluation,
            "web_result": web_result
        }

    if verbose:
        print("Agent: generando respuesta final con contexto web...")

    answer = generate_answer_with_context(
        query=user_query,
        context=web_result["context"],
        source_type="web"
    )

    return {
        "answer": answer,
        "source_type": "web",
        "rag_result": rag_result,
        "rag_evaluation": rag_evaluation,
        "web_result": web_result
    }

### Interfaz de respuesta

In [41]:
def ask_agent(question: str, show_sources: bool = True) -> None:
    result = agent_loop(
        user_query=question,
        verbose=True
    )

    print()
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print("Respuesta final")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(result["answer"])
    print()
    print("Fuente utilizada:", result["source_type"])

    if show_sources and result["source_type"] == "rag":
        print()
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print("Fuentes RAG utilizadas")
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(result["rag_sources_text"])

___

___

## Testeo de querys

### Querys

In [43]:
test_queries = [
    {
        "query": "What is renewable energy and what are its main sources?",
        "expected_case": "in_domain_good_rag_result"
    },
    {
        "query": "How does renewable energy help reduce greenhouse gas emissions?",
        "expected_case": "in_domain_good_rag_result"
    },
    {
        "query": "What are the main challenges for renewable energy deployment in the European Union?",
        "expected_case": "in_domain_good_rag_result"
    },
    {
        "query": "What is the best renewable energy source for a small restaurant in Hermosillo?",
        "expected_case": "in_domain_poor_rag_result"
    },
    {
        "query": "What are the main causes of commercial aircraft shortages in 2025?",
        "expected_case": "out_of_domain"
    }
]

In [44]:
for test in test_queries:
    print("QUERY:", test["query"])
    print("EXPECTED:", test["expected_case"])
    print()

    result = agent_loop(
        user_query=test["query"],
        verbose=True
    )

    print("SOURCE USED:", result["source_type"])
    print("ANSWER:")
    print(result["answer"])
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

QUERY: What is renewable energy and what are its main sources?
EXPECTED: in_domain_good_rag_result

Agent: buscando información en el RAG local...
Agent: evaluando si el contexto del RAG es suficiente...
Agent: el RAG tiene información suficiente.
Agent: generando respuesta final con contexto RAG...
SOURCE USED: rag
ANSWER:
Renewable energy es el energía proveniente de fuentes naturales que se regeneran continuamente y son abundantes, como la luz solar, el viento y el agua. Estas fuentes son sostenibles porque no se agotarán fácilmente con el tiempo de uso humano.

Las principales fuentes de energía renovable incluyen:

1. **Energía Solar**: Utiliza la radiación del sol para generar electricidad mediante paneles fotovoltaicos o sistemas térmicos que captan la calidez del sol.
2. **Energía Eólica**: Aprovecha el viento para hacer girar las aspas de turbinas, convirtiendo así la energía cinética del viento en electricidad.
3. **Energía Hidroeléctrica**: Genera electricidad utilizando la 